# Week 5 · Parte B — Agente del torneo Connect-4

**Studio:** [`README.md`](README.md) · **Notebook de la sesión A:** [`../../notebooks/week-05-minimax-alphabeta.ipynb`](../../notebooks/week-05-minimax-alphabeta.ipynb)

La sesión 5B *es* el torneo. El torneo exige una sola interfaz:

```python
def choose_move(board, time_budget_ms) -> int
```

Reglas duras: **500 ms por jugada** · sin procesos externos · sin red · sin libro de aperturas > 20 posiciones. Pasarse del presupuesto = derrota por forfeit.

Este notebook implementa las dos funciones de `starter.py`:

1. **`order_moves(board)`** — columnas legales, mejores primero (centro primero + orden *dinámico*: la mejor jugada de la iteración anterior va primero).
2. **`choose_move(board, time_budget_ms)`** — **iterative deepening**: profundidad 1, 2, 3, … guardando la mejor jugada de la última profundidad **completada** y sin pasarse del reloj. Se construye en dos versiones:
   - **v1** (`choose_move_basic`): sobre el `alphabeta` *dado* en `connect4.py`, tal como lo plantea `starter.py`.
   - **v2** (`choose_move`, la que va al torneo): negamax propio con la *misma* evaluación, victoria incremental, tabla de transposición y corte duro por tiempo.

Al final se corren los 7 tests de `test_alphabeta.py` contra estas funciones.

## 1 · Motor dado

`connect4.py` contiene el tablero, `minimax`, `alphabeta`, las órdenes estáticas y `play`, copiados del notebook de la sesión A. **No se modifica**: solo se importa.

In [1]:
import sys
import time
from math import inf
from pathlib import Path

# El notebook vive en studios/week-05/, junto a connect4.py y test_alphabeta.py.
STUDIO = Path.cwd()
if not (STUDIO / "connect4.py").exists():
    STUDIO = Path.cwd() / "studios" / "week-05"
sys.path.insert(0, str(STUDIO))

from connect4 import (Connect4, minimax, alphabeta, play,
                      MAX, MIN, ROWS, CENTER_FIRST)

## 2 · `order_moves` — orden de exploración

α-β solo alcanza su mejor caso O(b^(m/2)) — **el doble de profundidad** — si examina primero las jugadas fuertes.

- **Estático:** centro primero (`CENTER_FIRST = [3, 2, 4, 1, 5, 0, 6]`): las columnas centrales participan en más líneas de 4.
- **Dinámico:** con iterative deepening ya conocemos la mejor jugada de la profundidad anterior (`first`); ponerla primero hace que la ventana α-β se cierre de inmediato y el resto se pode.

`alphabeta` recibe `order` como una lista de prioridad que filtra por legalidad en cada nodo (`[c for c in order if c in moves]`), así que la misma lista sirve en todo el árbol.

In [2]:
def order_moves(board, first=None):
    # Columnas legales de `board`, mejores primero. `first` (opcional) es la mejor
    # jugada de la iteración anterior: se prueba antes que todas las demás.
    legal = board.legal_moves()
    order = [c for c in CENTER_FIRST if c in legal]
    if first is not None and first in order:
        order.remove(first)
        order.insert(0, first)
    return order


order_moves(Connect4()), order_moves(Connect4(), first=5)

([3, 2, 4, 1, 5, 0, 6], [5, 3, 2, 4, 1, 0, 6])

## 3 · Chequeos tácticos de un ply

Dos atajos baratos (≈7 `push`/`pop` cada uno) antes de buscar:

- **Ganar ya:** si alguna columna completa 4 en línea, jugarla. Como `evaluate()` devuelve +10 000 para *cualquier* victoria sin importar a qué profundidad ocurre, una búsqueda profunda podría preferir una victoria más lejana; este chequeo garantiza tomar la inmediata.
- **Bloquear ya:** si MIN gana en una columna en su próximo turno, taparla.

Esto es exactamente lo que el retador LLM falla en clase.

In [3]:
def winning_move(board, player):
    # Columna con la que `player` gana inmediatamente, o None.
    for c in order_moves(board):
        board.push(c, player)
        won = board.winner() == player
        board.pop(c)
        if won:
            return c
    return None

## 4 · Versión 1: iterative deepening sobre el `alphabeta` dado

Primero la versión que pide el `starter.py` al pie de la letra (`choose_move_basic`). El `alphabeta` dado no se puede interrumpir a la mitad, así que el control del tiempo se hace **entre profundidades**:

1. **Siempre hay una jugada legal:** `best` se inicializa antes del ciclo.
2. **No empezar lo que no se puede terminar:** se estima el costo de la siguiente profundidad multiplicando el tiempo de la última por el factor de crecimiento observado (`t_d / t_{d-1}`, con un mínimo de 2). Si `transcurrido + estimado` supera el margen de seguridad (85 % del presupuesto), se detiene.
3. **Se conserva la mejor jugada de la última profundidad completada**, y esa jugada va primero en el orden de la siguiente iteración (orden dinámico).

Extras:
- Si la búsqueda encuentra una victoria forzada (`value ≥ 10 000`) se detiene: más profundidad no cambia nada.
- Si a cierta profundidad todas las jugadas pierden (`value ≤ -10 000`), se mantiene la jugada de la profundidad anterior: el oponente podría no ver la línea ganadora.
- Quedan a lo sumo `42 − discos` jugadas; no tiene sentido buscar más hondo.

In [4]:
WIN_SCORE = 10_000
SAFETY = 0.85            # fracción del presupuesto que realmente usamos
MIN_GROWTH = 2.0         # crecimiento mínimo supuesto entre profundidades

# Estadísticas de la última llamada (para inspección / demostraciones).
last_stats = {}


def tactical_move(board):
    # Ganar ya si se puede; si no, bloquear la victoria inmediata de MIN.
    for player in (MAX, MIN):
        c = winning_move(board, player)
        if c is not None:
            return c
    return None


def choose_move_basic(board, time_budget_ms) -> int:
    t_start = time.perf_counter()
    deadline = t_start + time_budget_ms / 1000.0 * SAFETY

    order = order_moves(board)
    best = order[0]                               # regla 1: siempre legal
    last_stats.clear()
    last_stats.update(depth=0, nodes=0, value=None, reason="tactic", ms=0.0)

    tactic = tactical_move(board)                 # atajos de un ply
    if tactic is not None:
        last_stats["ms"] = (time.perf_counter() - t_start) * 1000
        return tactic

    remaining_plies = sum(ROWS - h for h in board.heights)
    prev_time = last_time = None
    total_nodes = 0

    for depth in range(1, remaining_plies + 1):
        now = time.perf_counter()
        if now >= deadline:
            last_stats["reason"] = "deadline"
            break
        # regla 2: ¿alcanza el tiempo para terminar esta profundidad?
        if last_time is not None:
            growth = max(MIN_GROWTH, last_time / prev_time) if prev_time else 2 * MIN_GROWTH
            if now + last_time * growth > deadline:
                last_stats["reason"] = "prediction"
                break

        counter = [0]
        t0 = time.perf_counter()
        value, move = alphabeta(board, depth, MAX, -inf, inf, counter,
                                order_moves(board, first=best))
        prev_time, last_time = last_time, max(time.perf_counter() - t0, 1e-6)
        total_nodes += counter[0]

        if move is None:
            break
        if value <= -WIN_SCORE and depth > 1:
            # Todas pierden contra juego perfecto: conservar la jugada anterior.
            last_stats["reason"] = "lost"
            break
        best = move                               # regla 3: última completada
        last_stats.update(depth=depth, nodes=total_nodes, value=value,
                          reason="exhausted")
        if value >= WIN_SCORE:
            last_stats["reason"] = "forced win"
            break

    last_stats["ms"] = (time.perf_counter() - t_start) * 1000
    return best

## 5 · Prueba rápida de la versión 1 — ¿qué profundidad alcanza con 500 ms?

In [5]:
def fresh():
    b = Connect4()
    b.push(3, MAX); b.push(3, MIN)
    return b


def report(agent, budget_ms=500):
    for name, b in [("vacío", Connect4()), ("dos discos al centro", fresh())]:
        move = agent(b, budget_ms)
        assert move in b.legal_moves()
        print(f"{name:>22}: col {move} · profundidad {last_stats['depth']} · "
              f"{last_stats['nodes']:,} nodos · {last_stats['ms']:.0f} ms · "
              f"paró por: {last_stats['reason']}")


report(choose_move_basic)

                 vacío: col 3 · profundidad 5 · 1,418 nodos · 94 ms · paró por: prediction
  dos discos al centro: col 3 · profundidad 5 · 2,331 nodos · 202 ms · paró por: prediction


**Límite de la versión 1.** Llega apenas a profundidad ~5 — menos que el agente de referencia del bracket (α-β plano a profundidad 6), y la predicción de tiempo puede fallar: si la siguiente profundidad cuesta más de lo estimado, no hay forma de cortarla y la jugada se pasa de 500 ms (forfeit). El costo viene del motor dado: cada nodo llama `winner()`, que recorre **todo** el tablero, y `evaluate()` vuelve a llamar `winner()`.

## 6 · Versión 2: búsqueda propia más rápida y *interrumpible*

El README lo sugiere para las parejas que terminan rápido: orden dinámico y **tabla de transposición**. La versión 2 mantiene exactamente la misma función de evaluación (material por ventanas + centro), pero:

| Mejora | Por qué |
|---|---|
| **Negamax** sobre una copia plana del tablero (42 celdas) | un solo caso en vez de MAX/MIN duplicados |
| **Victoria incremental:** solo se revisan las ventanas que contienen la ficha recién jugada | `winner()` completo por nodo era el cuello de botella |
| **Distancia a la victoria:** ganar vale `10 000 − ply` | prefiere la victoria más rápida y, si pierde, la derrota más lenta |
| **Tabla de transposición** con hash Zobrist | la misma posición por distinto orden de jugadas no se busca dos veces; su mejor jugada se prueba primero (orden dinámico en *cada* nodo, no solo en la raíz) |
| **Corte duro por tiempo:** cada 512 nodos se mira el reloj y se lanza `Timeout` | una profundidad a medias se descarta y se usa la última completada — ya no depende de adivinar |

In [6]:
from connect4 import COLS, WIN
import random

N_CELLS = ROWS * COLS
# Las 69 ventanas de 4 celdas (índice plano r*COLS + c), igual que Connect4._windows.
WINDOWS = []
for r in range(ROWS):
    for c in range(COLS):
        for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
            rr, cc = r + dr * (WIN - 1), c + dc * (WIN - 1)
            if 0 <= rr < ROWS and 0 <= cc < COLS:
                WINDOWS.append(tuple((r + dr * k) * COLS + (c + dc * k) for k in range(WIN)))
WINDOWS_AT = [[w for w in WINDOWS if i in w] for i in range(N_CELLS)]
CENTER_CELLS = [r * COLS + COLS // 2 for r in range(ROWS)]
WINDOW_SCORE = {1: 1, 2: 10, 3: 50}

_rng = random.Random(2026)
ZOBRIST = {(i, p): _rng.getrandbits(64) for i in range(N_CELLS) for p in (MAX, MIN)}

EXACT, LOWER, UPPER = 0, 1, 2
CHECK_EVERY = 511          # máscara: mirar el reloj cada 512 nodos


class Timeout(Exception):
    pass


class Searcher:
    def __init__(self, board, deadline):
        self.cells = [board.grid[r][c] for r in range(ROWS) for c in range(COLS)]
        self.heights = list(board.heights)
        self.deadline = deadline
        self.nodes = 0
        self.tt = {}
        self.key = 0
        for i, p in enumerate(self.cells):
            if p:
                self.key ^= ZOBRIST[i, p]

    # ---- utilidades -------------------------------------------------------
    def evaluate(self):
        # Misma heurística que Connect4.evaluate() (posición no terminal), desde MAX.
        cells, score = self.cells, 0
        for w in WINDOWS:
            mx = mn = 0
            for i in w:
                v = cells[i]
                if v == MAX:
                    mx += 1
                elif v == MIN:
                    mn += 1
            if mx and mn:
                continue
            if mx:
                score += WINDOW_SCORE[mx]
            elif mn:
                score -= WINDOW_SCORE[mn]
        for i in CENTER_CELLS:
            score += 3 * cells[i]
        return score

    def wins_at(self, i, p):
        cells = self.cells
        for w in WINDOWS_AT[i]:
            if cells[w[0]] == p and cells[w[1]] == p and cells[w[2]] == p and cells[w[3]] == p:
                return True
        return False

    def moves(self, tt_move):
        legal = [c for c in CENTER_FIRST if self.heights[c] < ROWS]
        if tt_move is not None and tt_move in legal:
            legal.remove(tt_move)
            legal.insert(0, tt_move)
        return legal

    # ---- negamax α-β con tabla de transposición --------------------------
    def negamax(self, depth, alpha, beta, color, ply):
        self.nodes += 1
        if not (self.nodes & CHECK_EVERY) and time.perf_counter() > self.deadline:
            raise Timeout

        alpha_orig = alpha
        entry = self.tt.get(self.key)
        tt_move = None
        if entry is not None:
            e_depth, e_flag, e_value, tt_move = entry
            if e_depth >= depth and ply > 0:
                if e_flag == EXACT:
                    return e_value, tt_move
                if e_flag == LOWER:
                    alpha = max(alpha, e_value)
                else:
                    beta = min(beta, e_value)
                if alpha >= beta:
                    return e_value, tt_move

        if depth == 0:
            return color * self.evaluate(), None
        moves = self.moves(tt_move)
        if not moves:
            return 0, None                                # tablero lleno: empate

        best, best_move = -inf, moves[0]
        cells, heights = self.cells, self.heights
        for c in moves:
            i = heights[c] * COLS + c
            cells[i] = color
            heights[c] += 1
            self.key ^= ZOBRIST[i, color]
            try:
                if self.wins_at(i, color):
                    value = WIN_SCORE - ply - 1           # victoria: cuanto antes, mejor
                else:
                    value = -self.negamax(depth - 1, -beta, -alpha, -color, ply + 1)[0]
            finally:
                self.key ^= ZOBRIST[i, color]
                heights[c] -= 1
                cells[i] = 0
            if value > best:
                best, best_move = value, c
            alpha = max(alpha, value)
            if alpha >= beta:
                break

        flag = UPPER if best <= alpha_orig else LOWER if best >= beta else EXACT
        self.tt[self.key] = (depth, flag, best, best_move)
        return best, best_move

**Chequeo de consistencia.** La evaluación rápida debe coincidir con `Connect4.evaluate()` del motor dado en posiciones no terminales; si no, la versión 2 estaría jugando *otro* juego.

In [7]:
rng = random.Random(0)
checked = 0
for _ in range(300):
    b, player = Connect4(), MAX
    for _ in range(rng.randrange(0, 30)):
        if b.winner() is not None or b.full():
            break
        b.push(rng.choice(b.legal_moves()), player)
        player = -player
    if b.winner() is None:
        assert Searcher(b, inf).evaluate() == b.evaluate()
        checked += 1
print(f"evaluación idéntica en {checked} posiciones aleatorias")

evaluación idéntica en 212 posiciones aleatorias


## 7 · `choose_move` final (versión 2)

Mismo contrato que la versión 1 (siempre legal · no pasarse del presupuesto · última profundidad completada), ahora con corte duro: si la profundidad `d` no termina a tiempo, se lanza `Timeout` y se devuelve la mejor jugada de `d − 1`. La tabla de transposición se conserva entre profundidades de una misma llamada, así que cada iteración empieza por la mejor jugada de la anterior en todos los nodos.

In [8]:
SAFETY_V2 = 0.90


def choose_move(board, time_budget_ms) -> int:
    t_start = time.perf_counter()
    deadline = t_start + time_budget_ms / 1000.0 * SAFETY_V2

    best = order_moves(board)[0]                  # regla 1: siempre legal
    last_stats.clear()
    last_stats.update(depth=0, nodes=0, value=None, reason="tactic", ms=0.0)

    tactic = tactical_move(board)
    if tactic is not None:
        last_stats["ms"] = (time.perf_counter() - t_start) * 1000
        return tactic

    searcher = Searcher(board, deadline)
    remaining_plies = sum(ROWS - h for h in board.heights)
    last_stats["reason"] = "exhausted"
    for depth in range(1, remaining_plies + 1):
        try:
            value, move = searcher.negamax(depth, -inf, inf, MAX, 0)
        except Timeout:
            last_stats["reason"] = "timeout"      # profundidad incompleta: se descarta
            break
        if move is not None:
            best = move                           # regla 3: última completada
        last_stats.update(depth=depth, value=value)
        if abs(value) >= WIN_SCORE - 42:
            last_stats["reason"] = "forced win" if value > 0 else "forced loss"
            break                                 # resultado demostrado
        if time.perf_counter() >= deadline:
            last_stats["reason"] = "deadline"
            break

    last_stats["nodes"] = searcher.nodes
    last_stats["ms"] = (time.perf_counter() - t_start) * 1000
    return best


report(choose_move)

                 vacío: col 3 · profundidad 8 · 30,208 nodos · 455 ms · paró por: timeout


  dos discos al centro: col 3 · profundidad 8 · 26,112 nodos · 454 ms · paró por: timeout


## 8 · Tests del studio

`test_alphabeta.py` hace `import starter as s` y llama `s.order_moves` / `s.choose_move`. Aquí se reemplaza ese módulo por las funciones de este notebook y se corren los **7 tests** (5 del agente + las 2 *garantías* del motor).

In [9]:
import types
import test_alphabeta as T

T.s = types.SimpleNamespace(order_moves=order_moves, choose_move=choose_move)

failed = 0
for t in T.TESTS:
    try:
        t()
    except AssertionError as e:
        print(f"  FAIL {t.__name__}: {e}")
        failed += 1
print(f"\n{len(T.TESTS) - failed}/{len(T.TESTS)} tests pass")

  ok  order_moves returns the legal columns, reordered


  ok  choose_move always returns a legal column
  ok  choose_move takes the immediate win (col 3)
  ok  choose_move blocks the immediate threat (col 3)


  ok  choose_move respected the budget (278 ms / 300 ms)


  ok  α-β returns minimax's value while expanding fewer nodes


  ok  horizon effect: the shallow agent loses to the deeper one

7/7 tests pass


## 9 · Partidas contra el agente de referencia

El bracket de referencia usa un α-β plano a **profundidad 6**. Cada versión juega con ambos colores. Como el agente siempre busca como `MAX`, cuando juega de `O` se le entrega el tablero con los colores invertidos.

Se registra también la **jugada más lenta** y la profundidad media alcanzada (límite: 500 ms).

In [10]:
def flipped(board):
    # Copia del tablero con MAX y MIN intercambiados.
    b = Connect4()
    b.grid = [[-v for v in row] for row in board.grid]
    b.heights = list(board.heights)
    return b


def reference_agent(board, player, depth=6):
    _, move = alphabeta(board, depth, player, -inf, inf, [0], CENTER_FIRST)
    return move


def match(agent, agent_plays, budget_ms=500):
    board, player, worst_ms, depths = Connect4(), MAX, 0.0, []
    while board.winner() is None and not board.full():
        if player == agent_plays:
            view = board if player == MAX else flipped(board)
            t0 = time.perf_counter()
            move = agent(view, budget_ms)
            worst_ms = max(worst_ms, (time.perf_counter() - t0) * 1000)
            if last_stats["depth"]:
                depths.append(last_stats["depth"])
        else:
            move = reference_agent(board, player)
        board.push(move, player)
        player = -player
    return board.winner(), board, worst_ms, depths


names = {MAX: "X (primero)", MIN: "O (segundo)"}
for label, agent in [("v1 · alphabeta dado", choose_move_basic),
                     ("v2 · negamax + TT", choose_move)]:
    for side in (MAX, MIN):
        w, board, worst, depths = match(agent, side)
        result = ("GANA el agente" if w == side else "empate" if w is None
                  else "gana la referencia")
        avg = sum(depths) / len(depths) if depths else 0
        flag = "  <-- FORFEIT (>500 ms)" if worst > 500 else ""
        print(f"{label} como {names[side]}: {result} · profundidad media {avg:.1f} · "
              f"jugada más lenta {worst:.0f} ms{flag}")
        print(board, "\n")

v1 · alphabeta dado como X (primero): gana la referencia · profundidad media 5.8 · jugada más lenta 363 ms
|O O X X . O O|
|X X X O . O O|
|O O X O O X X|
|X X O X X O O|
|X O X O O X O|
|X X O X X X O|
 0 1 2 3 4 5 6 



v1 · alphabeta dado como O (segundo): gana la referencia · profundidad media 5.8 · jugada más lenta 597 ms  <-- FORFEIT (>500 ms)
|. . X O . X .|
|. O O X X X X|
|. X X X O X O|
|. O O X X O X|
|O O X O X X O|
|O X O X O O O|
 0 1 2 3 4 5 6 



v2 · negamax + TT como X (primero): GANA el agente · profundidad media 8.6 · jugada más lenta 458 ms
|. . . . . . .|
|. . O O . . .|
|. . X X O . .|
|. . O X X . .|
|. . X O X X .|
|. . O X O O X|
 0 1 2 3 4 5 6 



v2 · negamax + TT como O (segundo): GANA el agente · profundidad media 10.4 · jugada más lenta 457 ms
|O X O O . O .|
|X X X O . X .|
|O O X O O X .|
|X X O X X O .|
|O O X O O X O|
|X X O X X O X|
 0 1 2 3 4 5 6 



## 10 · El retador LLM

En la sesión 5B entra al bracket un agente LLM con el prompt del plan (`{board}` = `str(board)`, `{legal}` = `board.legal_moves()`). Se usa el cliente del curso, `aicourse.llm.LLM` (backend `ollama`, modelo por defecto `qwen2.5:3b`, con caché en `.llm_cache/`).

Qué medimos, según el README:
- **Legalidad:** la respuesta se valida; si no es una columna legal se reintenta (hasta 3 veces, con otra semilla). Si nunca acierta, juega la primera columna legal y se cuenta como fallo.
- **Victorias inmediatas perdidas** y **amenazas no bloqueadas:** cada una se guarda con el tablero **exacto** para el Failure Atlas.
- **Tiempo por jugada** contra el agente α-β.

> Requisito: `ollama serve` corriendo con el modelo descargado. La ruta de `aicourse` se configura abajo (`AICOURSE_PATH`).

In [11]:
import os
import re

AICOURSE_PATH = Path(os.environ.get(
    "AICOURSE_PATH", r"C:\Users\andyv\Downloads\cmp4004-week0\cmp4004-week0"))
sys.path.insert(0, str(AICOURSE_PATH))       # carpeta que CONTIENE el paquete aicourse

from aicourse.llm import LLM, backends_available

print({k: v for k, v in backends_available().items()})
model = LLM(backend="ollama", cache_dir=str(STUDIO / ".llm_cache"))
print(f"backend: {model.backend} · modelo: {model.model}")

PROMPT = '''Connect-4. You are 'X'. Columns numbered 0-6, left to right.
Board (bottom row last):
{board}
Legal columns: {legal}
Reply with one column number and nothing else.'''

{'ollama': 'serving (3 model(s))', 'manual': 'always available', 'echo': 'always available (testing only — not a model)'}
backend: ollama · modelo: qwen2.5:3b


In [12]:
MAX_TRIES = 3
llm_log = {"moves": 0, "illegal_replies": 0, "gave_up": 0, "seconds": 0.0,
           "missed_wins": [], "missed_blocks": []}


def parse_column(text):
    m = re.search(r"\d+", text or "")
    return int(m.group()) if m else None


def llm_agent(board, time_budget_ms=None):
    # Agente LLM con la misma interfaz que choose_move (juega como MAX = 'X').
    legal = board.legal_moves()
    prompt = PROMPT.format(board=str(board), legal=legal)
    move = None
    for attempt in range(MAX_TRIES):
        # intento 0: greedy y reproducible; reintentos: muestreo con otra semilla
        r = (model.complete(prompt) if attempt == 0
             else model.complete(prompt, temperature=0.7, seed=attempt))
        llm_log["seconds"] += r.elapsed
        if r.error:
            raise RuntimeError(f"el LLM falló: {r.error}")
        col = parse_column(r.text)
        if col in legal:
            move = col
            break
        llm_log["illegal_replies"] += 1
    if move is None:
        llm_log["gave_up"] += 1
        move = legal[0]
    llm_log["moves"] += 1

    # Failure Atlas: ¿había una victoria o un bloqueo obligatorio y no lo jugó?
    win, block = winning_move(board, MAX), winning_move(board, MIN)
    if win is not None and move != win:
        llm_log["missed_wins"].append((str(board), win, move))
    elif win is None and block is not None and move != block:
        llm_log["missed_blocks"].append((str(board), block, move))
    last_stats.clear()
    last_stats.update(depth=0)
    return move

### 10.1 · Las dos posiciones tácticas de los tests

Las mismas que exige `test_alphabeta.py` a nuestro agente: ganar en la columna 3 y bloquear en la columna 3.

In [13]:
for name, b in [("ganar ya (col 3)", T._immediate_win_for_max()),
                ("bloquear ya (col 3)", T._immediate_threat_against_max())]:
    m_llm = llm_agent(b)
    m_ab = choose_move(b, 500)
    print(f"{name}:  LLM -> {m_llm} · α-β -> {m_ab}")
    print(b, "\n")

ganar ya (col 3):  LLM -> 2 · α-β -> 3
|. . . . . . .|
|. . . . . . .|
|. . . . . . .|
|. . . . . . .|
|. . . . . . .|
|X X X . O O O|
 0 1 2 3 4 5 6 



bloquear ya (col 3):  LLM -> 3 · α-β -> 3
|. . . . . . .|
|. . . . . . .|
|. . . . . . .|
|. . . O . . .|
|. . . O . . .|
|X X X O . . .|
 0 1 2 3 4 5 6 



### 10.2 · LLM contra nuestro agente (ambos colores)

In [14]:
def duel(agent_x, agent_o, budget_ms=500):
    # agent_x juega como MAX ('X'); agent_o como MIN ('O', recibe el tablero invertido).
    board, player = Connect4(), MAX
    times = {MAX: [], MIN: []}
    while board.winner() is None and not board.full():
        agent = agent_x if player == MAX else agent_o
        view = board if player == MAX else flipped(board)
        t0 = time.perf_counter()
        move = agent(view, budget_ms)
        times[player].append(time.perf_counter() - t0)
        assert move in board.legal_moves()
        board.push(move, player)
        player = -player
    return board.winner(), board, times


def avg_ms(ts):
    return 1000 * sum(ts) / len(ts) if ts else 0.0


for label, ax, ao, llm_side in [("LLM (X) vs α-β (O)", llm_agent, choose_move, MAX),
                                ("α-β (X) vs LLM (O)", choose_move, llm_agent, MIN)]:
    w, board, times = duel(ax, ao)
    winner = "empate" if w is None else ("LLM" if w == llm_side else "α-β")
    print(f"{label}: gana {winner} · "
          f"LLM {avg_ms(times[llm_side]):.0f} ms/jugada · "
          f"α-β {avg_ms(times[-llm_side]):.0f} ms/jugada")
    print(board, "\n")

LLM (X) vs α-β (O): gana α-β · LLM 2318 ms/jugada · α-β 413 ms/jugada
|. . . . X X .|
|. . . . O O .|
|. O . . X O .|
|. O . O O X .|
|. O . O X X X|
|. O X X X O X|
 0 1 2 3 4 5 6 



α-β (X) vs LLM (O): gana α-β · LLM 2312 ms/jugada · α-β 324 ms/jugada
|. . . . . . .|
|. . . O . . .|
|. . . X X . .|
|. . X X O . .|
|. . X O O . .|
|. X O X O . .|
 0 1 2 3 4 5 6 



### 10.3 · Resumen y Failure Atlas

Cada entrada guarda el tablero exacto (visto desde el LLM, que siempre es `X`), la jugada correcta y la que jugó, para que el caso se pueda reproducir.

In [15]:
n = llm_log["moves"]
print(f"jugadas del LLM: {n}")
print(f"respuestas ilegales (antes de reintentar): {llm_log['illegal_replies']}")
print(f"jugadas donde agotó los reintentos: {llm_log['gave_up']}")
print(f"victorias inmediatas perdidas: {len(llm_log['missed_wins'])}")
print(f"amenazas no bloqueadas: {len(llm_log['missed_blocks'])}")
print(f"tiempo de inferencia (sin caché): {llm_log['seconds']:.1f} s en total")

for kind, entries in [("VICTORIA PERDIDA", llm_log["missed_wins"]),
                      ("BLOQUEO OMITIDO", llm_log["missed_blocks"])]:
    for board_str, correct, played in entries:
        print(f"\n[{kind}] correcta: col {correct} · jugó: col {played}")
        print(board_str)

jugadas del LLM: 19
respuestas ilegales (antes de reintentar): 0
jugadas donde agotó los reintentos: 0
victorias inmediatas perdidas: 1
amenazas no bloqueadas: 2
tiempo de inferencia (sin caché): 47.9 s en total

[VICTORIA PERDIDA] correcta: col 3 · jugó: col 2
|. . . . . . .|
|. . . . . . .|
|. . . . . . .|
|. . . . . . .|
|. . . . . . .|
|X X X . O O O|
 0 1 2 3 4 5 6

[BLOQUEO OMITIDO] correcta: col 1 · jugó: col 6
|. . . . X X .|
|. . . . O O .|
|. . . . X O .|
|. O . O O X .|
|. O . O X X .|
|. O X X X O X|
 0 1 2 3 4 5 6

[BLOQUEO OMITIDO] correcta: col 1 · jugó: col 3
|. . . . . . .|
|. . . . . . .|
|. . . O O . .|
|. . O O X . .|
|. . O X X . .|
|. . X O X . .|
 0 1 2 3 4 5 6


## 11 · Conclusiones

- **Pruning no es aproximación:** el test de garantía confirma que α-β devuelve *exactamente* el valor de minimax con muchos menos nodos.
- **El orden importa:** centro primero + mejor jugada previa primero (en la v2, en *cada* nodo gracias a la tabla de transposición) hacen que cada iteración pode más, y eso se traduce en profundidad extra dentro de los mismos 500 ms.
- **Iterative deepening = agente *anytime*:** una profundidad fija o desperdicia presupuesto o pierde por tiempo. La v1 solo puede *predecir* si la siguiente profundidad cabe; la v2 la **interrumpe** y vuelve a la última completada, así que nunca se pasa del reloj.
- **Misma evaluación, más profundidad:** la v2 usa exactamente la heurística del motor dado; toda la diferencia contra la referencia viene de buscar más hondo.
- **El LLM no busca:** reconocer la descripción de un juego no es calcular dentro de él. Por eso no toma victorias inmediatas ni bloquea amenazas que un agente de ~40 líneas encuentra a profundidad 1, y además es mucho más lento. Ruoss et al. (2024) juegan ajedrez a nivel gran maestro sin búsqueda en inferencia, pero eso no contradice lo anterior: *la búsqueda no desapareció, se movió al entrenamiento*.
- **Horizonte:** el agente de profundidad fija pierde no por jugar mal, sino por no ver la amenaza que aparece un ply después de su horizonte. *Depth beats knowledge at this scale.*